# 预训练

当调用搭建好的`MiniMindForCausalLM`类实例化一个模型之后，模型的参数是随机的，这个阶段的模型没有任何语言能力，无法进行有意义的文本生成或理解。

**预训练**使用大规模的无监督语料对模型进行训练，使其具备“理解和生成自然语言”的基础能力，为后续的**微调**提供一个好的起点。

---
本节详细解析了 MiniMind 项目中的核心训练代码，涵盖数据处理、训练工具库以及预训练主程序。

要理解Minimind的训练方法，我们要重点关注三个文件：dataset/lm_dataset.py，trainer/trainer_utils.py，trainer/train_pretrain.py。

# 1 数据处理：`lm_dataset.py`
该文件定义了不同训练阶段所需的 `Dataset` 类，负责将原始文本转换为模型可读的 Tensor。

首先我们看看用于预训练的文本长什么样子。

查看minimind的预训练数据集`/dataset/pretrain_t2t_mini.jsonl`中的第一条数据

In [1]:
import json
pretrain_dataset_path=r'/Users/shelly/Desktop/minimind/minimind/dataset/pretrain_t2t_mini.jsonl'
with open(pretrain_dataset_path, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        data = json.loads(line.strip())
        break
        
print(data.keys()) # dict_keys(['text'])
print(data)

dict_keys(['text'])
{'text': '给我生成一首有关秋天的诗歌。秋日早晨，清风拂面。\n金色的叶子，似火在燃烧。\n露珠晶莹，如珍珠般美丽。\n秋的气息，弥漫在空气中。\n余音袅袅，如鸟儿的歌唱。\n美丽的秋天，是大自然的馈赠。帮我想一些创意，给即将到来的公司年会准备节目。一些节目比如能否请一位表演者为我们表演一曲钢琴曲，或者请一位小提琴手为大家演奏一首古典曲目。如果想要画面更具有视觉冲击力，可以安排一个魔术师或者杂技演员的表演。另外，也可以设计一些小游戏或者有奖竞猜来增添活动的趣味性，这些小游戏可以和公司的文化、发展历程等相关。请问给我讲一个清净的法则。无为而治是一项清净的法则。即在处理问题时，不要强行干预，反而尽可能地减少干预，坚持自然的发展趋势。比如，让植物自然生长，照顾它们就行了，不需要过度地修剪和整齐地排列。类似地，让人们自由发展，而不是通过繁琐的管制、政策干涉等方式，去要求、指导人们的行为。这种无为而治的法则旨在维持一个有秩序的、和平的治理状态，却不需要大规模的干预和指挥。'}


这份 `pretrain_hq.jsonl` 数据非常有趣。虽然它放在 Pretrain 阶段，但它的内容其实是 **QA 对话**。

这在业界被称为 **"Instruction Pre-training" (指令预训练)** 或者是用 Chat 数据进行 **Continued Pre-training**。
- **为什么这么做？**
  - 一般的 Pretrain 数据是维基百科、书籍（长篇大论）。
  - 这份数据是“一问一答”的短文本。
  - 把这些数据放到 Pretrain 阶段（不 Mask），是为了让模型在**大规模无监督学习**的时候，就提前熟悉“有人提问，紧接着就是回答”这种**文本概率分布**。

## 1.0 文件头与预处理函数

In [ ]:
from torch.utils.data import Dataset
import torch
import json
import os
import random
from datasets import load_dataset, Features, Sequence, Value

# 关闭 tokenizer 的并行提示，避免多进程数据加载时出现额外警告或干扰。
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def pre_processing_chat(conversations, add_system_ratio=0.2):
    """对多轮对话样本做轻量预处理。

    主要目的有两个：
    1. 保留工具调用样本的原始结构，避免破坏 tool-use 标注。
    2. 对普通对话按概率补一个 system 提示词，让模型见到更多样的系统指令前缀，
       提升泛化能力。
    """
    # 如果样本里已经包含 tools 字段，说明这是工具调用类数据。
    # 这类样本通常对消息结构要求比较严格，因此不做任何改动，直接返回。
    if any(conv.get('tools') for conv in conversations):
        return conversations

    # 系统提示词池。
    # 这里混合了中文和英文，目的不是“翻译”，而是让模型对不同风格的 system prompt 更鲁棒。
    SYSTEM_PROMPTS = [
        "你是一个知识丰富的AI，尽力为用户提供准确的信息。",
        "你是minimind，一个小巧但有用的语言模型。",
        "你是一个专业的AI助手，请提供有价值的回答。",
        "你是minimind，请尽力帮助用户解决问题。",
        "你是一个可靠的AI，请给出准确的回答。",
        "You are a helpful AI assistant.",
        "You are minimind, a lightweight intelligent assistant.",
        "You are a friendly chatbot. Please answer the user's questions carefully.",
        "You are a knowledgeable AI. Try your best to provide accurate information.",
        "You are minimind, a small but useful language model."
    ]

    # 如果第一条消息不是 system，就按概率随机补一个。
    # 这样做的目的是让模型在训练时既见到“有 system”的样本，
    # 也见到“没有 system”的样本，减少对固定前缀格式的依赖。
    if conversations[0].get('role') != 'system':
        if random.random() < add_system_ratio:
            return [{'role': 'system', 'content': random.choice(SYSTEM_PROMPTS)}] + conversations

    return conversations


def post_processing_chat(prompt_content, empty_think_ratio=0.2):
    """对拼接完成的 prompt 做后处理。

    当前主要做一件事：
    - 对 `<think>\n\n</think>\n\n` 这种空思考标签，按概率移除。
      这样可以减少训练数据中大量无意义的空思考片段，
      让模型更专注于真正的推理与回答内容。
    """
    # empty_think_ratio=0.2 表示保留 20%，移除 80%。
    # 这里用 random.random() > empty_think_ratio，因此默认会在 80% 的情况下删除空 think 段。
    if '<think>\n\n</think>\n\n' in prompt_content and random.random() > empty_think_ratio:
        prompt_content = prompt_content.replace('<think>\n\n</think>\n\n', '')
    return prompt_content

## 1.1 `PretrainDataset` (预训练数据集)

- **用途**：用于无监督预训练（Causal Language Modeling）。
- **逻辑**：
  1. 读取 JSONL 文件中的文本。
  2. 使用 Tokenizer 进行编码，并在首尾添加 `BOS` (Start) 和 `EOS` (End) 标记。
  3. **Padding**：将长度不足 `max_length` 的数据填充 `pad_token_id`。
  4. **Label 生成**：在预训练中，标签（Labels）与输入（Input IDs）完全一致（自回归任务），但在计算 Loss 时，模型会自动忽略 Pad 部分（通过设置 Padding 的 label 为 -100）。

In [3]:
from torch.utils.data import Dataset
import torch
import json
import os
import random
from datasets import load_dataset, Features, Sequence, Value

/Users/shelly/miniconda3/envs/minimind/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class PretrainDataset(Dataset):
    """预训练数据集。

    作用：
    - 读取纯文本语料 `text`
    - 将文本转成固定长度 token 序列
    - 构造自回归语言模型训练所需的 `input_ids` 和 `labels`

    这里的标签通常和输入 token 序列一致，只是把 padding 位置置为 -100，
    这样损失函数会忽略 pad 部分。
    """
    def __init__(self, data_path, tokenizer, max_length=512):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length # 指的是 token 长度

        # 从 json / jsonl 文件中读取预训练样本。
        # split='train' 表示把这份文件当成训练集。
        self.samples = load_dataset('json', data_files=data_path, split='train')

    def __len__(self):
        # 返回数据集样本数，DataLoader 会据此决定一个 epoch 有多少条数据。
        return len(self.samples)

    def __getitem__(self, index):
        # 取出单条样本。
        sample = self.samples[index]

        # 先做普通分词，不添加额外 special tokens。
        # max_length - 2 是为了给 BOS 和 EOS 留出位置。
        tokens = self.tokenizer(
            str(sample['text']),
            add_special_tokens=False,
            max_length=self.max_length - 2,
            truncation=True
        ).input_ids

        # 手动补上 BOS / EOS。
        # 这样模型能明确知道序列起点和结束位置。
        tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]

        # 不足 max_length 的部分用 pad 补齐，保证 batch 内长度一致。
        input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))
        input_ids = torch.tensor(input_ids, dtype=torch.long)

        # labels 一般直接复制 input_ids。
        # 训练时模型内部会做 next-token 预测的 shift。
        labels = input_ids.clone()

        # padding 部分不参与 loss，因此设成 -100。
        # PyTorch 的交叉熵 loss 会忽略 -100。
        labels[input_ids == self.tokenizer.pad_token_id] = -100

        return input_ids, labels

构建数据加载器：

In [13]:
from torch.utils.data import DataLoader
#导入 DataLoader，它的作用是把 Dataset 里的单条样本组织成一个个 batch
from transformers import AutoTokenizer


max_length=512 #每条样本会被处理成固定长度 512
data_path=r'/Users/shelly/Desktop/minimind/minimind/dataset/pretrain_t2t_mini.jsonl'
tokenizer = AutoTokenizer.from_pretrained("./model")
train_ds = PretrainDataset(data_path, tokenizer, max_length)
#构造预训练数据集对象，
#它内部会读取 pretrain_t2t_mini.jsonl，每条样本会返回两个张量：input_ids，labels

#把 train_ds 包装成可迭代的数据加载器
train_loader = DataLoader(
    train_ds,       #数据来源，也就是你刚创建的 dataset。
    batch_size=2,   #每次取 2 条样本组成一个 batch。
    pin_memory=True,#可以加快 CPU 到 GPU 的拷贝
    drop_last=False,#如果最后一个 batch 不够 2 条，也保留
    shuffle=False,  #不打乱顺序，按原始数据顺序读取
    num_workers=0,  #数据只在主进程里加载，不开额外子进程
)

查看数据总量以及数据的维度信息：

In [ ]:
print(len(train_ds)) #train_loader所加载的样本总数
print(len(train_loader)) # 一个batch中的样本数
#batch_size=1 时，这两个数才会相等
for item in train_loader:
    print([i.shape for i in item]) # [torch.Size([2, 511]), torch.Size([2, 511]), torch.Size([2, 511])]
    break

635119
1270238
[torch.Size([2, 512]), torch.Size([2, 512])]


## 1.3 `DPODataset` (偏好优化数据集)

DPO 偏好数据集，读取 chosen/rejected 两条对话，构造对应的输入序列和 loss mask。

- **用途**：用于 DPO (Direct Preference Optimization) 训练。
- **逻辑**：同时加载 `chosen`（人类偏好）和 `rejected`（人类拒绝）的数据对。返回成对的数据 `(x_chosen, y_chosen)` 和 `(x_rejected, y_rejected)` 及其对应的掩码。

这是用于DPO的数据处理，后面再讲。

In [ ]:
class DPODataset(Dataset):
    """DPO 偏好数据集。

    作用：
    - 读取一条样本中的 chosen / rejected 两个答案
    - 分别构造 token 序列和 loss mask
    - 供 DPO 这类偏好优化算法使用

    注意：
    这个类本身不计算 DPO loss，
    它只负责把样本整理成训练代码需要的格式。
    """
    def __init__(self, file_path, tokenizer, max_length=4096):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length

        # pad_token_id 可能为空，因此这里做一个兜底。
        # 虽然当前代码里 self.padding 没有直接用到，但保留它有助于后续统一处理。
        self.padding = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

        # 用于识别 assistant 回复段。
        # 这里和 SFTDataset 保持一致，方便复用“只监督回答部分”的逻辑。
        self.bos_id = tokenizer(f'{tokenizer.bos_token}assistant\n', add_special_tokens=False).input_ids
        self.eos_id = tokenizer(f'{tokenizer.eos_token}\n', add_special_tokens=False).input_ids

        self.samples = load_dataset('json', data_files=file_path, split='train')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]

        # chosen 表示模型偏好的回答，rejected 表示不偏好的回答。
        # 它们都是完整对话列表。
        chosen = sample['chosen']
        rejected = sample['rejected']

        # 将 chosen / rejected 分别套入 chat template。
        chosen_prompt = self.tokenizer.apply_chat_template(
            chosen, tokenize=False, add_generation_prompt=False
        )
        chosen_prompt = post_processing_chat(chosen_prompt)

        rejected_prompt = self.tokenizer.apply_chat_template(
            rejected, tokenize=False, add_generation_prompt=False
        )
        rejected_prompt = post_processing_chat(rejected_prompt)

        # 分别 tokenize，并强制 padding 到同一长度。
        chosen_encoding = self.tokenizer(
            chosen_prompt, truncation=True, max_length=self.max_length, padding='max_length'
        )
        rejected_encoding = self.tokenizer(
            rejected_prompt, truncation=True, max_length=self.max_length, padding='max_length'
        )

        chosen_input_ids = chosen_encoding['input_ids']
        # mask 只标记 assistant 回复区域，prompt 部分为 0。
        chosen_loss_mask = self.generate_loss_mask(chosen_input_ids)

        rejected_input_ids = rejected_encoding['input_ids']
        rejected_loss_mask = self.generate_loss_mask(rejected_input_ids)

        # 这里做 next-token 形式的输入输出对齐：
        # x = 当前 token 序列
        # y = 右移一个位置后的目标 token 序列
        # mask 也要同步右移，确保和 y 对齐
        x_chosen = torch.tensor(chosen_input_ids[:-1], dtype=torch.long)
        y_chosen = torch.tensor(chosen_input_ids[1:], dtype=torch.long)
        mask_chosen = torch.tensor(chosen_loss_mask[1:], dtype=torch.long)

        x_rejected = torch.tensor(rejected_input_ids[:-1], dtype=torch.long)
        y_rejected = torch.tensor(rejected_input_ids[1:], dtype=torch.long)
        mask_rejected = torch.tensor(rejected_loss_mask[1:], dtype=torch.long)

        return {
            'x_chosen': x_chosen,
            'y_chosen': y_chosen,
            'mask_chosen': mask_chosen,
            'x_rejected': x_rejected,
            'y_rejected': y_rejected,
            'mask_rejected': mask_rejected
        }

    def generate_loss_mask(self, input_ids):
        """生成 loss mask。

        规则：
        - prompt / system / user 部分：0，不参与 loss
        - assistant 回复部分：1，参与 loss

        这个 mask 通常用于只统计回答区域的 logprob 或 loss，
        避免偏好优化时把 prompt 部分也算进去。
        """
        loss_mask = [0] * len(input_ids)
        i = 0

        while i < len(input_ids):
            if input_ids[i:i + len(self.bos_id)] == self.bos_id:
                start = i + len(self.bos_id)
                end = start

                while end < len(input_ids):
                    if input_ids[end:end + len(self.eos_id)] == self.eos_id:
                        break
                    end += 1

                # 回复段的 token 标记为 1。
                # 这里把 EOS 一并纳入，避免回复边界被漏掉。
                for j in range(start, min(end + len(self.eos_id), self.max_length)):
                    loss_mask[j] = 1

                i = end + len(self.eos_id) if end < len(input_ids) else len(input_ids)
            else:
                i += 1

        return loss_mask

# 2. 基础设施层：`trainer_utils.py`

该文件提供了训练所需的“脚手架”功能，特别是针对分布式训练和模型状态管理的封装。

# 3. 核心训练引擎：`train_pretrain.py`

这是整个预训练阶段的训练控制中心，负责组装模型、数据和优化器，并执行训练循环。

它负责把 PretrainDataset 读出来的纯文本数据喂给 MiniMindForCausalLM，完成下面这些事情：
- 初始化分布式训练环境
- 设置随机种子，保证尽量可复现
- 加载模型、tokenizer 和已有权重
- 构建数据集、DataLoader 和 batch 采样器
- 使用混合精度和梯度累积进行训练
- 定期打印日志、保存 checkpoint
- 支持断点续训时跳过已经训练过的 batch

## 3.1 核心函数：train_epoch

[train_epoch]负责真正的训练循环，也就是每个 batch 的：
- 前向传播
- loss 计算
- 反向传播
- 梯度累积
- 梯度裁剪
- 参数更新
- 日志打印
- checkpoint 保存

if __name__ == "__main__": 

是启动和编排流程，负责：
- 初始化分布式 
- 设置随机种子
- 加载模型和数据
- 构造优化器
- 恢复断点
- 进入 epoch 循环

预训练代码和常规的模型训练代码几乎没有区别，核心代码段如下：

In [ ]:
def train_epoch(epoch, loader, iters, start_step=0, wandb=None):
    """
    训练一个 epoch。

    这个函数负责训练过程中的核心逻辑：
    - 从 DataLoader 里取 batch
    - 前向传播
    - 计算损失
    - 反向传播
    - 梯度累积
    - 梯度裁剪
    - 学习率更新
    - 日志打印
    - 保存模型

    注意：
    这个函数并不显式接收 model / optimizer / scaler / args，
    而是直接使用外层 main 里定义的全局变量。
    """
    # 记录这个 epoch 开始的时间，用于估算训练速度和剩余时间。
    start_time = time.time()

    # 记录最后一个处理到的 step，便于 epoch 结束后处理未完成的梯度累积。
    last_step = start_step

    # loader 每次返回一个 batch。
    # 这里的 batch 结构来自 PretrainDataset:
    #   input_ids, labels
    for step, (input_ids, labels) in enumerate(loader, start=start_step + 1):
        # 把数据搬到目标设备上，比如 cuda:0、cuda:1 或 cpu。
        input_ids = input_ids.to(args.device)
        labels = labels.to(args.device)

        # 记录当前 step，后面判断是否需要补一次优化器更新会用到。
        last_step = step

        # 计算当前 step 的学习率。
        # epoch * iters + step 表示“全局训练进度”。
        lr = get_lr(epoch * iters + step, args.epochs * iters, args.learning_rate)

        # PyTorch 优化器支持逐个 param_group 修改学习率。
        # 这里通常只有一个 param_group，但写法更通用。
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # 混合精度上下文：
        # - CPU 上没有必要使用 autocast
        # - GPU 上则启用 autocast 以提升速度、降低显存占用
        with autocast_ctx:
            # 前向传播。
            # model(input_ids, labels=labels) 通常会返回一个包含 loss / aux_loss / logits 的结果对象。
            res = model(input_ids, labels=labels)
            #是`MiniMindForCausalLM`这个类的实例。

            # 总损失 = 主损失 + 辅助损失。
            # 主损失来自语言建模目标，aux_loss 可能来自模型内部的额外正则或 MoE 辅助项。
            loss = res.loss + res.aux_loss

            # 梯度累积：
            # 如果 accumulation_steps > 1，就对 loss 求平均，
            # 让多个小 batch 的梯度累积效果接近一个大 batch。
            loss = loss / args.accumulation_steps

        # 使用 GradScaler 对 loss 做缩放（混合精度）后反向传播。
        # 这样在 float16 下更稳定。
        scaler.scale(loss).backward()

        # 每累积 accumulation_steps 个 batch，再更新一次参数。
        if step % args.accumulation_steps == 0:
            # 先反缩放梯度，避免后面的梯度裁剪受缩放影响。
            scaler.unscale_(optimizer)

            # 裁剪梯度范数，防止梯度爆炸。
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

            # 更新参数。
            scaler.step(optimizer)

            # 更新 scaler 的内部状态。
            scaler.update()

            # 清空梯度，为下一轮累积做准备。
            optimizer.zero_grad(set_to_none=True)

        # 每隔 log_interval 步打印一次日志，或者跑到最后一个 step 时也打印。
        if step % args.log_interval == 0 or step == iters:
            spend_time = time.time() - start_time

            # 这里乘回 accumulation_steps，是因为前面把 loss 除掉了。
            current_loss = loss.item() * args.accumulation_steps

            # aux_loss 如果不存在，就记成 0。
            current_aux_loss = res.aux_loss.item() if res.aux_loss is not None else 0.0

            # 主 logits loss = 总 loss - aux_loss。
            current_logits_loss = current_loss - current_aux_loss

            # 取当前优化器里的学习率。
            current_lr = optimizer.param_groups[-1]['lr']

            # 估计剩余时间，单位是分钟。
            # 这里是一个粗略估计，不是严格精确值。
            eta_min = spend_time / max(step - start_step, 1) * (iters - step) // 60

            Logger(
                f'Epoch:[{epoch + 1}/{args.epochs}]({step}/{iters}), '
                f'loss: {current_loss:.4f}, '
                f'logits_loss: {current_logits_loss:.4f}, '
                f'aux_loss: {current_aux_loss:.4f}, '
                f'lr: {current_lr:.8f}, '
                f'epoch_time: {eta_min:.1f}min'
            )

            # 如果启用了 wandb / swanlab，就把指标同步上去。
            if wandb:
                wandb.log({
                    "loss": current_loss,
                    "logits_loss": current_logits_loss,
                    "aux_loss": current_aux_loss,
                    "learning_rate": current_lr,
                    "epoch_time": eta_min
                })

        # 到达保存点时保存模型。
        # 只有主进程才允许保存，避免 DDP 多进程同时写文件。
        if (step % args.save_interval == 0 or step == iters) and is_main_process():
            model.eval()

            # 如果使用 MoE，就在文件名中加 _moe，便于区分模型结构。
            moe_suffix = '_moe' if lm_config.use_moe else ''
            ckp = f'{args.save_dir}/{args.save_weight}_{lm_config.hidden_size}{moe_suffix}.pth'

            # DDP 包装后，真正的模型在 model.module 里。
            raw_model = model.module if isinstance(model, DistributedDataParallel) else model

            # 如果模型被 torch.compile 包过一层，真实对象可能在 _orig_mod 里。
            # getattr(..., default) 这一写法可以兼容没有 _orig_mod 的情况。
            raw_model = getattr(raw_model, '_orig_mod', raw_model)

            # 取出当前模型参数。
            state_dict = raw_model.state_dict()

            # 保存权重时转成 half precision 并放到 CPU，
            # 可以减小文件体积，也更方便后续加载。
            torch.save({k: v.half().cpu() for k, v in state_dict.items()}, ckp)

            # 另存一个“可断点恢复”的完整状态文件：
            # 里面不仅有 model，还包含 optimizer、scaler、epoch、step 等信息。
            lm_checkpoint(
                lm_config,
                weight=args.save_weight,
                model=model,
                optimizer=optimizer,
                scaler=scaler,
                epoch=epoch,
                step=step,
                wandb=wandb,
                save_dir='../checkpoints'
            )

            # 切回训练模式。
            model.train()

            # 主动删除临时变量，减少内存占用。
            del state_dict

        # 及时释放本 step 的中间变量，减少内存压力。
        del input_ids, labels, res, loss

    # 如果 epoch 最后还剩下没凑满 accumulation_steps 的梯度，
    # 这里再补一次 optimizer.step()，避免最后一段梯度被丢掉。
    if last_step > start_step and last_step % args.accumulation_steps != 0:
        scaler.unscale_(optimizer)# 取消scale，准备裁剪梯度（clip操作要求原始梯度）
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)# 裁剪梯度，防止爆炸
        scaler.step(optimizer)# 执行优化器步进
        scaler.update()# 更新scaler内部状态
        optimizer.zero_grad(set_to_none=True)# 清空梯度，准备下一次累计